# Skip-gram with Negative Sampling (Wikipedia Corpus)

This notebook is converted from skipgram.py and separated based on the 5 steps questions.

## 1) Use a Wikipedia article as the dataset

- Selected corpus: AI winter article from Wikipedia
- Requirement note: article should be reasonably long (a few thousand words)
- This section loads NLTK resources and downloads/extracts the article text.

In [1]:
import re
import json
import random
from typing import List, Tuple, Dict

import requests
from bs4 import BeautifulSoup
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from gensim.models import Word2Vec
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

WIKI_URL = "https://en.wikipedia.org/wiki/AI_winter"
RANDOM_SEED = 42


def ensure_nltk():
    resources = ["punkt", "punkt_tab"]
    for r in resources:
        try:
            nltk.data.find(f"tokenizers/{r}")
        except LookupError:
            nltk.download(r)


def fetch_wikipedia_article(url: str) -> str:
    headers = {
        "User-Agent": "Mozilla/5.0 (compatible; SGNS-Gundam-Training/1.0)"
    }
    resp = requests.get(url, headers=headers, timeout=30)
    resp.raise_for_status()

    soup = BeautifulSoup(resp.text, "html.parser")
    content_div = soup.find("div", {"id": "mw-content-text"})
    if content_div is None:
        raise ValueError("Could not find Wikipedia article content.")

    paragraphs = content_div.find_all(["p", "li"])
    text_blocks = []
    for p in paragraphs:
        txt = p.get_text(" ", strip=True)
        if txt:
            text_blocks.append(txt)

    text = "\n".join(text_blocks)
    text = re.sub(r"\[[0-9]+\]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
ensure_nltk()

raw_text = fetch_wikipedia_article(WIKI_URL)
print("Downloaded characters:", len(raw_text))
print("Preview:", raw_text[:300], "...")

Downloaded characters: 39674
Preview: Artificial general intelligence Intelligent agent Recursive self-improvement Planning Computer vision General game playing Knowledge representation Natural language processing Robotics AI safety Machine learning Symbolic Deep learning Bayesian networks Evolutionary algorithms Hybrid intelligent syst ...


## 2) Preprocess text from the selected corpus

This section lowercases, tokenizes, cleans punctuation/digits, and keeps meaningful tokens.

In [2]:
def preprocess_text(text: str) -> List[List[str]]:
    sentences = sent_tokenize(text)

    processed = []
    for sent in sentences:
        sent = sent.lower()
        sent = re.sub(r"[^a-z0-9\-\s]", " ", sent)
        sent = re.sub(r"\s+", " ", sent).strip()
        if not sent:
            continue

        tokens = word_tokenize(sent)

        cleaned = []
        for tok in tokens:
            tok = tok.strip("-")
            if not tok:
                continue
            if tok.isdigit():
                continue
            if len(tok) < 2:
                continue
            cleaned.append(tok)

        if len(cleaned) >= 3:
            processed.append(cleaned)

    return processed


def corpus_stats(sentences: List[List[str]]) -> Dict[str, int]:
    flat = [w for s in sentences for w in s]
    vocab = set(flat)
    return {
        "num_sentences": len(sentences),
        "num_tokens": len(flat),
        "vocab_size": len(vocab),
    }


sentences = preprocess_text(raw_text)
stats = corpus_stats(sentences)
print(stats)
print("Sample tokens from first sentence:", sentences[0][:20])

{'num_sentences': 271, 'num_tokens': 5223, 'vocab_size': 1839}
Sample tokens from first sentence: ['artificial', 'general', 'intelligence', 'intelligent', 'agent', 'recursive', 'self-improvement', 'planning', 'computer', 'vision', 'general', 'game', 'playing', 'knowledge', 'representation', 'natural', 'language', 'processing', 'robotics', 'ai']


## 3) Train a Skip-gram with Negative Sampling model

This section trains `Word2Vec` with `sg=1` (skip-gram) and negative sampling.

In [3]:
def train_sgns(sentences: List[List[str]]) -> Word2Vec:
    model = Word2Vec(
        sentences=sentences,
        vector_size=100,
        window=5,
        min_count=1,
        workers=4,
        sg=1,
        negative=10,
        epochs=200,
        sample=1e-3,
        alpha=0.025,
        min_alpha=0.0007,
        seed=RANDOM_SEED,
    )
    return model


model = train_sgns(sentences)
print("Vocabulary size learned:", len(model.wv))
print("Word2Vec vector_size:", model.vector_size)
print("Word2Vec window:", model.window)

Vocabulary size learned: 1839
Word2Vec vector_size: 100
Word2Vec window: 5


## 4) Evaluate embeddings using a small test set

This section uses relatedness and analogy mini test sets with AI-winter-focused words.

In [4]:
def has_word(model: Word2Vec, word: str) -> bool:
    return word in model.wv.key_to_index


def cosine(model: Word2Vec, w1: str, w2: str) -> float:
    v1 = model.wv[w1].reshape(1, -1)
    v2 = model.wv[w2].reshape(1, -1)
    return float(cosine_similarity(v1, v2)[0][0])


def evaluate_relatedness(model: Word2Vec, test_pairs: List[Tuple[str, str, float]]):
    covered = []
    for w1, w2, score in test_pairs:
        if has_word(model, w1) and has_word(model, w2):
            sim = cosine(model, w1, w2)
            covered.append((w1, w2, score, sim))
    return {
        "covered_items": covered,
        "coverage": len(covered),
        "total": len(test_pairs),
    }


def evaluate_analogies(model: Word2Vec, analogies: List[Tuple[str, str, str, str]]):
    covered = 0
    correct = 0
    details = []

    for a, b, c, d in analogies:
        if all(has_word(model, w) for w in [a, b, c, d]):
            covered += 1
            preds = model.wv.most_similar(positive=[b, c], negative=[a], topn=5)
            predicted_words = [w for w, _ in preds]
            hit = d in predicted_words
            correct += int(hit)
            details.append({
                "analogy": f"{a}:{b}::{c}:?",
                "expected": d,
                "predictions": predicted_words,
                "correct_in_top5": hit,
            })

    accuracy = correct / covered if covered else float("nan")
    return {
        "coverage": covered,
        "total": len(analogies),
        "accuracy_top5": accuracy,
        "details": details,
    }


probe_words = [
    "ai", "winter", "research", "funding", "expert",
    "systems", "neural", "network", "machine", "learning",
]

relatedness_test = [
    ("ai", "artificial", 0.95),
    ("ai", "intelligence", 0.95),
    ("expert", "systems", 0.92),
    ("machine", "learning", 0.94),
    ("neural", "network", 0.93),
    ("research", "funding", 0.70),
    ("winter", "funding", 0.60),
    ("symbolic", "reasoning", 0.78),
    ("ai", "kitchen", 0.05),
    ("neural", "tractor", 0.03),
    ("expert", "weather", 0.20),
    ("research", "hype", 0.40),
]

analogy_test = [
    ("machine", "learning", "neural", "network"),
    ("expert", "systems", "neural", "networks"),
    ("symbolic", "reasoning", "machine", "learning"),
    ("funding", "research", "hype", "expectations"),
]

rel_results = evaluate_relatedness(model, relatedness_test)
analogy_results = evaluate_analogies(model, analogy_test)

print("Relatedness coverage:", f"{rel_results['coverage']}/{rel_results['total']}")
print("Analogy coverage:", f"{analogy_results['coverage']}/{analogy_results['total']}")
print("Analogy top-5 accuracy:", analogy_results["accuracy_top5"])

Relatedness coverage: 9/12
Analogy coverage: 4/4
Analogy top-5 accuracy: 0.0


## 5) Report nearest neighbors, similarity scores, and test-set performance

This section prints nearest neighbors, relatedness details, analogy predictions, direct similarities, and saves the trained model.

In [5]:
def print_top_neighbors(model: Word2Vec, words: List[str], topn: int = 8):
    print("\n=== Nearest Neighbors ===")
    for word in words:
        if has_word(model, word):
            neighbors = model.wv.most_similar(word, topn=topn)
            print(f"\n{word}:")
            for neigh, score in neighbors:
                print(f"  {neigh:20s} {score:.4f}")
        else:
            print(f"\n{word}: [OOV]")


print_top_neighbors(model, probe_words, topn=8)

print("\n=== Relatedness Test Set ===")
print(f"Coverage: {rel_results['coverage']}/{rel_results['total']}")
for w1, w2, gold, pred in rel_results["covered_items"]:
    print(f"{w1:10s} - {w2:10s} | gold={gold:.2f} pred={pred:.4f}")

print("\n=== Analogy Test Set ===")
print(f"Coverage: {analogy_results['coverage']}/{analogy_results['total']}")
print(f"Top-5 accuracy: {analogy_results['accuracy_top5']}")
for item in analogy_results["details"]:
    print(json.dumps(item, ensure_ascii=False))

print("\n=== Direct Similarity Checks ===")
check_pairs = [
    ("ai", "intelligence"),
    ("machine", "learning"),
    ("expert", "systems"),
    ("ai", "kitchen"),
]
for w1, w2 in check_pairs:
    if has_word(model, w1) and has_word(model, w2):
        print(f"{w1:10s} <-> {w2:10s}: {cosine(model, w1, w2):.4f}")
    else:
        print(f"{w1:10s} <-> {w2:10s}: OOV")

model.save("exercise_5_skipgram_sgns.model")
print("\nSaved model to: exercise_5_skipgram_sgns.model")


=== Nearest Neighbors ===

ai:
  literacy             0.5083
  slop                 0.4978
  center               0.4903
  veganism             0.4823
  effect               0.4737
  eu                   0.4480
  act                  0.4479
  elections            0.4407

winter:
  communications       0.6548
  veganism             0.6541
  nuclear              0.6540
  anthropomorphism     0.6492
  arms                 0.6484
  similar              0.6382
  dates                0.6364
  slop                 0.6335

research:
  undirected           0.5360
  united               0.5320
  decrease             0.5145
  basic                0.5072
  kingdom              0.5006
  complete             0.4947
  pure                 0.4933
  strings              0.4910

funding:
  cuts                 0.5953
  brutally             0.5925
  eviscerating         0.5864
  survived             0.5673
  division             0.5560
  sci                  0.5491
  cutback              0.5487
  procur